# HealthBench — two boards over one exam

Can a fusion of open-weights models improve on a strong single model across
[HealthBench](https://openai.com/index/healthbench/) Professional conversations?

The Engine serves this exam as **two** boards. Same conversations pool, same
physician-written rubrics, same pinned Judge — they differ in exactly two places:

| | `healthbench-worst30` | `healthbench-professional` |
|---|---|---|
| Conversations asked | the 157 hardest (the 30% top models score worst on) | all 525 |
| Final score | plain average, **negatives kept** | the **official** average, floored at 0 |
| Answers | "how does this do on the hard tail?" | "how does this compare to published numbers?" |

Per-case scoring is identical on both: satisfying a rubric item adds its points, tripping a
safety item subtracts them, so one case can score below zero. The boards only disagree on
what to do with that at the end. On the hardest 157, flooring at 0 would flatten every
entrant to 0.00 — so worst30 keeps the negative. The full board floors it, because that is
what published HealthBench figures do.

<img src="assets/healthbench-worst30-benchmark.svg" width="900"
  alt="HealthBench worst-30 at a glance: 157 hardest conversations, physician-written
  rubrics where penalties subtract, unclamped case scores, raw mean keeps negatives"/>

## 0. Before running

From a terminal:

```bash
screamingface prepare healthbench  # first run only: download pinned Benchmark assets
screamingface up             # start Gateway :9105, Scoreboard :9106, and Engine :9108
screamingface status
```

Use `screamingface logs` to inspect startup failures and `screamingface down` when finished. Stack
management stays outside the notebook so **Run All** never starts or stops local services.

In [ ]:
import screamingface as sf

sf.connect()

## 1. Run the hard board with 1 model

`limit=1` runs a single Case — a cheap rehearsal that exercises the whole pipeline. Drop
the argument to sit the whole exam.

In [ ]:
PARAMS = {"max_tokens": 32768, "temperature": 0.0}

deepseek = sf.Model(
    model="openrouter/deepseek/deepseek-v4-pro",
    params=PARAMS,
)

In [ ]:
deepseek_report = sf.evaluate(deepseek, benchmark="healthbench-worst30", limit=1)
deepseek_report

## 2. Define the Fusion with open source models and evaluate it

In [ ]:
qwen = sf.Model(
    model="openrouter/qwen/qwen3.8-2.4t-a95b",
    params=PARAMS,
)
glm = sf.Model(
    model="openrouter/z-ai/glm-5.2",
    params=PARAMS,
)

In [ ]:
SYNTHESIS_PROMPT = (
    "You are synthesising the single best response to a health-related "
    "conversation by combining independent responses from a panel of models. "
    "A physician-authored rubric will grade your output per criterion — "
    "accuracy, completeness, safety, and appropriate hedging all matter; so "
    "does avoiding harmful or false content.\n\n"
    "Procedure:\n"
    "1. Read every panel response carefully.\n"
    "2. Identify what each contributes that the others miss (facts, safety "
    "caveats, follow-up questions, care-seeking guidance).\n"
    "3. Produce ONE unified response to the final user turn that:\n"
    "   - Keeps every clinically correct, relevant point\n"
    "   - Drops anything inaccurate, unsafe, or speculative\n"
    "   - Resolves disagreements by favouring the safer, better-supported claim\n"
    "   - Stays concise — verbosity is not rewarded\n"
    "4. Do not introduce claims no panel member made.\n\n"
    "Output: the unified response only, no preamble, no JSON wrapper."
)

kimi = sf.Model(
    model="openrouter/moonshotai/kimi-k3",
    params=PARAMS,
    prompt=SYNTHESIS_PROMPT,
)

best_open_source = sf.Fusion(
    members=[deepseek, qwen, glm], name="best_open_source", synthesizer=kimi
)

In [ ]:
worst30_report = sf.evaluate(best_open_source, benchmark="healthbench-worst30", limit=1)
worst30_report

## 3. Run the same Fusion on the full exam

Nothing about the Candidate changes — only the board it sits. This one asks all 525
conversations and reports the official HealthBench score, so its number is the one to put
beside a published figure.

Two things worth knowing before dropping `limit`:

- A full run costs roughly **3.3x** a full worst-30% run per candidate (525 conversations
  instead of 157, each with one Judge call per rubric item).
- The score is floored at 0. A candidate that trips enough safety items lands at 0.00
  here while still ranking above another entrant on the worst-30% board — that is the
  clip doing its job, not a bug.

In [ ]:
professional_report = sf.evaluate(best_open_source, benchmark="healthbench-professional", limit=1)
professional_report

## 4. Send the scores to the Scoreboard

Each board has its own Leaderboard, so a Candidate is submitted to each separately.
Publication takes the evaluated `CandidateResult` and submits the Benchmark's **native
score** exactly as the Engine graded it — fractional or negative values included — and the
Scoreboard stores and ranks it without recalculating. Opt-in so **Run All** never changes
the public Leaderboard.

In [ ]:
PUBLISH_RESULT = False

submissions = (
    [
        sf.leaderboards.submit(report.candidates.only)
        for report in (worst30_report, professional_report)
    ]
    if PUBLISH_RESULT
    else None
)
submissions